> **Ported notebook.** This notebook was originally a ``midas-ff-pipeline`` walkthrough. ``midas-ff-pipeline`` is deprecated as of 0.4.0; the FF workflow now lives in ``midas-pipeline`` with ``--scan-mode ff``. All API + CLI calls below have been retargeted; behaviour is unchanged. The legacy package will be removed in 1.0.0.


# Stage diagnostics — peek between stages

Runs each pipeline stage individually so you can plot intermediate
outputs (peaks, spots, indexed grains) before continuing. Useful for
debugging real datasets where the standard pipeline drops out without
obvious reason.

Assumes you already have a synthetic or real dataset ready (see
``01_smoke_walkthrough.ipynb`` for synthesis).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from midas_pipeline import Pipeline, PipelineConfig, ScanGeometry

## 1. Inputs

In [ ]:
params_file = Path("/tmp/midas_pipeline_smoke/sim/Parameters.txt")
zarr_path   = Path("/tmp/midas_pipeline_smoke/sim/midas_pipeline_synth.analysis.MIDAS.zip")
result_dir  = Path("/tmp/midas_pipeline_smoke/diag")

device = "cpu"
n_cpus = 8

pipe = Pipeline(
    config=PipelineConfig(
    scan=ScanGeometry.ff(),
        result_dir=str(result_dir),
        params_file=str(params_file),
        zarr_path=str(zarr_path),
        n_cpus=n_cpus,
        device=device,
        dtype="float64",
    ),
)

## 2. HKL + peakfit + merge — count peaks per ring

In [ ]:
pipe.run_hkl()
pipe.run_peakfit()
pipe.run_merge_overlaps()
pipe.run_calc_radius()

input_all = pipe.layer_dir / "InputAll.csv"
if input_all.exists():
    peaks = pd.read_csv(input_all)
    print(f"Total peaks: {len(peaks):,}")
    print(peaks.head())

## 3. Transforms + bin — Spots.bin scatter (Y-Z lab)

In [ ]:
pipe.run_transforms()
pipe.run_cross_det_merge()
pipe.run_binning()

spots = np.fromfile(pipe.layer_dir / "Spots.bin", dtype=np.float64).reshape(-1, 9)
print(f"n spots: {spots.shape[0]:,}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(spots[:, 0], spots[:, 1], s=1, alpha=0.4)
ax.set_aspect("equal")
ax.set_xlabel("Y_lab (um)"); ax.set_ylabel("Z_lab (um)")
ax.set_title(f"Spots.bin — {spots.shape[0]:,} obs")
plt.show()

## 4. Indexing — number of seeds with non-zero result

In [ ]:
ix = pipe.run_indexing()
print(f"  attempted: {ix.n_seeds_attempted}")
print(f"  indexed:   {ix.n_seeds_indexed}")

# Distribution of confidence (column 14 of IndexBest.bin)
ib = np.fromfile(ix.index_best_bin, dtype=np.float64).reshape(-1, 15)
ib = ib[ib[:, 14] > 0]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ib[:, 14], bins=40)
ax.set_xlabel("Confidence (col 14)"); ax.set_ylabel("Count")
ax.set_title("IndexBest.bin confidence distribution")
plt.show()

## 5. Refine + process-grains — final grain map

In [ ]:
pipe.run_refine()
pg = pipe.run_process_grains()
print(f"  grains found: {pg.n_grains}")

df = pipe.layer_result.grains_df()
print(df.shape)
df.head()

In [ ]:
if {"X", "Y"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df["X"], df["Y"], s=20, c=df.get("Confidence", "k"), cmap="viridis")
    ax.set_xlabel("X (um)"); ax.set_ylabel("Y (um)")
    ax.set_title(f"{len(df)} grains")
    ax.set_aspect("equal")
    plt.show()